# L3P Value Contrastive Loss Experiment on Kaggle

This notebook runs the PointMaze experiment comparing the original L3P value regression against the new InfoNCE negative-sampling loss for `V(g1, g2)`.

How to use on Kaggle:

1. Create a Kaggle Dataset from this repository, or add an existing dataset that contains the repo files.
2. Attach that dataset to this notebook.
3. Run all cells. The first cell copies the repo to `/kaggle/working/latent_landmarks` because `/kaggle/input` is read-only.
4. Start with `SMOKE = True`. For real comparison, set `SMOKE = False` and run one seed pair per Kaggle session, then repeat for more seeds.

The notebook is time-limit aware: it saves full training-state checkpoints periodically, stops jobs gracefully before the session limit if configured, and can resume from previous Kaggle outputs attached as input datasets.


In [ ]:
from pathlib import Path
import os
import shutil

def looks_like_repo(root: Path) -> bool:
    return (root / "l3p" / "config.py").exists() and (root / "scripts" / "train_pointmaze.py").exists()

def find_repo() -> Path:
    cwd = Path.cwd()
    if looks_like_repo(cwd):
        return cwd

    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not base.exists():
            continue
        for l3p_dir in base.rglob("l3p"):
            root = l3p_dir.parent
            if looks_like_repo(root):
                return root

    raise FileNotFoundError(
        "Could not find the latent_landmarks repo. Attach a Kaggle Dataset containing "
        "the repo, or upload the repo files into this notebook session."
    )

src = find_repo().resolve()
dst = Path("/kaggle/working/latent_landmarks").resolve() if Path("/kaggle/working").exists() else src

if src != dst:
    shutil.copytree(
        src,
        dst,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns(
            ".git", "logs", "*.pt", "__pycache__", ".pytest_cache", ".ipynb_checkpoints"
        ),
    )

os.chdir(dst)
print("Repo source:", src)
print("Working repo:", Path.cwd())
print("Files ok:", looks_like_repo(Path.cwd()))


In [ ]:
import importlib.util
import subprocess
import sys

required_modules = ["numpy", "torch", "matplotlib", "pandas"]
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "matplotlib", "pandas"
    ])

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("This repo trains on CPU by default; Kaggle CPU is enough for PointMaze smoke tests.")


## Sanity Check

Run the synthetic unit tests first. This verifies the replay negatives, the contrastive loss helper, and the original disabled path.

In [ ]:
RUN_TESTS = True

if RUN_TESTS:
    subprocess.run([sys.executable, "tests/test_modules.py"], check=True)


## Experiment Config

`SMOKE = True` is only for checking the pipeline. For an actual result, set `SMOKE = False`. On Kaggle, run one seed pair per session, commit the output, then attach the previous output dataset and resume or run the next seed.

In [ ]:
RUN_TRAINING = True

# True: about 30k env steps, good for debugging only.
# False: use FULL_STEPS below, better for measuring performance.
SMOKE = True

FULL_STEPS = 500_000
# Kaggle CPU can be slow; one baseline+contrastive pair per session is safer.
# Repeat later with SEEDS = [1], [2], ... and aggregate the logs.
SEEDS = [0]
EVAL_EPISODES = 10 if SMOKE else 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Safety knobs for Kaggle's runtime cap.
CHECKPOINT_EVERY = 5_000 if SMOKE else 50_000
SAVE_TRAINING_STATE = True
RESUME_IF_CHECKPOINT_EXISTS = True
# Full mode gives each variant about half of an 11-hour usable budget.
TIME_LIMIT_HOURS_PER_JOB = None if SMOKE else 5.25
RUN_TAG = "smoke" if SMOKE else f"{FULL_STEPS // 1000}k"

VALUE_CONTRASTIVE_LAMBDA = 0.1
VALUE_CONTRASTIVE_TEMPERATURE = 1.0
N_VALUE_NEGATIVES = 4
NEGATIVE_SAMPLING_STRATEGY = "random"  # "random" or "cross_episode"

OUT_DIR = Path("/kaggle/working/l3p_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def import_previous_outputs():
    """Copy prior Kaggle output checkpoints/logs into OUT_DIR for resume."""
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return []

    copied = []
    for pattern in ["pm_*_s*.pt", "pm_*_s*.log"]:
        for src_path in input_root.rglob(pattern):
            dst_path = OUT_DIR / src_path.name
            if dst_path.exists():
                continue
            shutil.copy2(src_path, dst_path)
            copied.append(dst_path)
    return copied

previous = import_previous_outputs()

print("mode:", "smoke" if SMOKE else "full")
print("seeds:", SEEDS)
print("run tag:", RUN_TAG)
print("device:", DEVICE)
print("checkpoint every:", CHECKPOINT_EVERY)
print("time limit per job:", TIME_LIMIT_HOURS_PER_JOB)
print("output dir:", OUT_DIR)
print("imported previous artifacts:", len(previous))


In [ ]:
def stream_command(cmd):
    print("\n$", " ".join(map(str, cmd)))
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

def train_one(variant: str, seed: int):
    assert variant in {"baseline", "contrastive"}
    log_path = OUT_DIR / f"pm_{RUN_TAG}_{variant}_s{seed}.log"
    save_path = OUT_DIR / f"pm_{RUN_TAG}_{variant}_s{seed}.pt"

    cmd = [
        sys.executable,
        "scripts/train_pointmaze.py",
        "--seed", str(seed),
        "--eval-episodes", str(EVAL_EPISODES),
        "--device", DEVICE,
        "--log-file", str(log_path),
        "--save", str(save_path),
        "--save-every", str(CHECKPOINT_EVERY),
    ]

    if SAVE_TRAINING_STATE:
        cmd.append("--save-training-state")
    if RESUME_IF_CHECKPOINT_EXISTS and save_path.exists():
        cmd.extend(["--load", str(save_path)])
    if TIME_LIMIT_HOURS_PER_JOB is not None:
        cmd.extend(["--time-limit-hours", str(TIME_LIMIT_HOURS_PER_JOB)])

    if SMOKE:
        cmd.append("--short")
    else:
        cmd.extend(["--steps", str(FULL_STEPS)])

    if variant == "contrastive":
        cmd.extend([
            "--value-contrastive",
            "--value-contrastive-lambda", str(VALUE_CONTRASTIVE_LAMBDA),
            "--value-contrastive-temperature", str(VALUE_CONTRASTIVE_TEMPERATURE),
            "--n-value-negatives", str(N_VALUE_NEGATIVES),
            "--negative-sampling-strategy", NEGATIVE_SAMPLING_STRATEGY,
        ])

    stream_command(cmd)
    return log_path, save_path


## Run Baseline vs Contrastive

This runs matched seeds. The only intentional difference is `--value-contrastive` and its hyperparameters.

In [ ]:
artifacts = []

if RUN_TRAINING:
    for seed in SEEDS:
        artifacts.append(("baseline", seed, *train_one("baseline", seed)))
        artifacts.append(("contrastive", seed, *train_one("contrastive", seed)))

artifacts


## Parse Logs and Plot

The main metric is long-horizon evaluation success rate. In smoke mode, this plot only checks that both variants run; do not over-interpret it.

In [ ]:
import re

PROG = re.compile(r"\[\s*(\d+)\s+steps")
EVAL = re.compile(r"eval success rate \(long-horizon test\):\s*([-\d.]+)")
FINAL = re.compile(r"Final long-horizon test success rate:\s*([-\d.]+)")

def parse_log(path: Path):
    variant = "contrastive" if "contrastive" in path.stem else "baseline"
    seed_match = re.search(r"_s(\d+)", path.stem)
    seed = int(seed_match.group(1)) if seed_match else -1
    rows = []
    last_step = 0

    with path.open() as f:
        for line in f:
            m = PROG.search(line)
            if m:
                last_step = int(m.group(1))
                continue
            e = EVAL.search(line)
            if e:
                rows.append({
                    "variant": variant,
                    "seed": seed,
                    "step": last_step,
                    "success": float(e.group(1)),
                    "kind": "periodic_eval",
                    "log": str(path),
                })
                continue
            final = FINAL.search(line)
            if final:
                rows.append({
                    "variant": variant,
                    "seed": seed,
                    "step": last_step,
                    "success": float(final.group(1)),
                    "kind": "final_eval",
                    "log": str(path),
                })
    return rows

rows = []
for log_path in sorted(OUT_DIR.glob("pm_*_s*.log")):
    rows.extend(parse_log(log_path))

df = pd.DataFrame(rows)
display(df.tail(20))

if df.empty:
    print("No eval rows found yet. Run the training cell first.")
else:
    periodic = df[df["kind"] == "periodic_eval"].copy()
    summary = df.groupby(["variant", "seed"])["success"].max().reset_index(name="best_success")
    display(summary)

    fig, ax = plt.subplots(figsize=(9, 5))
    for variant, color in [("baseline", "tab:blue"), ("contrastive", "tab:orange")]:
        sub = periodic[periodic["variant"] == variant]
        if sub.empty:
            continue

        for seed, seed_df in sub.groupby("seed"):
            ax.plot(seed_df["step"], seed_df["success"], color=color, alpha=0.25, linewidth=1)

        agg = sub.groupby("step")["success"].agg(["mean", "std", "count"]).reset_index()
        ax.plot(agg["step"], agg["mean"], marker="o", color=color, label=variant)
        if (agg["count"] > 1).any():
            std = agg["std"].fillna(0.0)
            ax.fill_between(agg["step"], agg["mean"] - std, agg["mean"] + std, color=color, alpha=0.15)

    ax.set_title("PointMaze long-horizon eval success")
    ax.set_xlabel("Environment steps")
    ax.set_ylabel("Success rate")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()


## Optional: Re-evaluate Saved Checkpoints

Use this after training if you want a cleaner final estimate with more evaluation episodes. This can take extra time.

In [ ]:
RUN_EXTRA_EVAL = False
EXTRA_EVAL_EPISODES = 50

if RUN_EXTRA_EVAL:
    for ckpt in sorted(OUT_DIR.glob("pm_*_s*.pt")):
        print("\nEvaluating", ckpt.name)
        stream_command([
            sys.executable,
            "scripts/eval.py",
            "--load", str(ckpt),
            "--episodes", str(EXTRA_EVAL_EPISODES),
        ])


## How to Decide Whether Contrastive Helps

Use these checks:

- Compare matched seeds: baseline seed 0 vs contrastive seed 0, seed 1 vs seed 1, and so on.
- Focus on long-horizon eval success, not only training loss.
- Do not trust `SMOKE = True` for conclusions; it is only a pipeline test.
- If contrastive is unstable, try `VALUE_CONTRASTIVE_LAMBDA = 0.05` before changing anything else.
- If results are noisy, increase `EVAL_EPISODES` and number of seeds.
